<a href="https://colab.research.google.com/github/sahradede/Realtime-Video-Stabilization/blob/main/realtime_stabilization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-time video stabilization

Causal stabilization — the pipeline never looks at future frames, so it runs on
a live stream with no added latency. Built for a Jetson Orin Nano, with a
synchronized stereo pair as the next step.

Everything lives in cells, so nothing is lost when the Colab session ends.
Sections A–E are the main run; the appendix keeps the experiments that led to
the current settings.

| section | what it does |
|---|---|
| A. Setup | definitions, run once |
| B. Data | three benchmark clips + settings |
| C. Run | process all clips with one config |
| D. Results | table, charts, videos |
| E. Deployment | what changes on the Jetson |
| Appendix | parameter sweeps and diagnostics |


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## A. Setup

Run these three once per session.


### A1. Core stabilizer

This is the only part that ships to the Jetson. Everything below it exists to test it.


In [ ]:
# Core stabilizer: corner-based budget, soft limiting,
# grid-distributed features, confidence weighting.
"""
Causal video stabilization for live streams.

Design notes
------------
1) No future frames. process() only ever sees what has already arrived.
   Trajectory smoothing is an EMA (IIR), not a sliding window, so the
   output frame is produced immediately with zero added latency.
2) Sources are pluggable. VideoFileSource for offline work today,
   CsiCameraSource on the Jetson tomorrow; nothing else changes.
3) Built with stereo in mind. Every frame carries a timestamp, a camera
   id and the warp that was applied. process() also accepts an
   external_correction so a future sync layer can force both cameras to
   use the *same* correction -- stabilizing two cameras independently
   would break the stereo geometry.
4) Speed. Motion is estimated on a downscaled grayscale frame; rotation,
   translation and the crop-zoom all go into a single warpAffine.
"""
from __future__ import annotations
import math, os, time
from dataclasses import dataclass
from typing import Iterator, Optional, Tuple, List
import cv2
import numpy as np


@dataclass
class Frame:
    image: np.ndarray
    timestamp: float
    index: int
    cam_id: int = 0


@dataclass
class Motion:
    dx: float = 0.0
    dy: float = 0.0
    da: float = 0.0          # radians
    valid: bool = False
    n_inliers: int = 0
    def as_array(self):
        return np.array([self.dx, self.dy, self.da], dtype=np.float64)


@dataclass
class StabilizedFrame:
    image: np.ndarray
    timestamp: float
    index: int
    cam_id: int
    correction: np.ndarray
    motion: Motion
    warp: np.ndarray
    n_tracks: int
    proc_ms: float


@dataclass
class StabConfig:
    proc_width: int = 480          # motion-estimation width; biggest speed lever
    max_corners: int = 200
    quality_level: float = 0.01
    min_distance: int = 12
    block_size: int = 3
    redetect_interval: int = 15
    min_tracks: int = 60
    lk_win: int = 21
    lk_levels: int = 3
    use_fb_check: bool = True      # forward-backward check (~20% slower)
    fb_threshold: float = 1.0
    ransac_thresh: float = 3.0
    min_inliers: int = 12
    # Defaults come from the benchmark sweep (see the appendix notebook).
    # adaptive=True was measured to fire almost constantly and wreck the
    # smoothing, so it is off by default. It may still help on footage with
    # fast sustained panning -- check visually before turning it back on.
    smooth_alpha: float = 0.90     # 0.85 = responsive, 0.97 = very smooth
    adaptive: bool = False
    crop_ratio: float = 0.12       # crop per side; zoom = 1/(1-2r)
    grid: int = 4                  # feature distribution grid
    conf_inliers: int = 40         # inlier count for full confidence
    windup_leak: float = 0.25      # anti-windup leak rate
    max_angle_deg: float = 3.0
    safety: float = 0.95           # unused since the corner check replaced it
    num_threads: int = 0
    interpolation: int = cv2.INTER_LINEAR
    border_mode: int = cv2.BORDER_REPLICATE


class FrameSource:
    def read(self) -> Optional[Frame]:
        raise NotImplementedError
    def release(self): pass
    def __iter__(self):
        while True:
            f = self.read()
            if f is None: break
            yield f


class VideoFileSource(FrameSource):
    """Reads a file but behaves like a camera: no seeking, no look-ahead."""
    def __init__(self, path, cam_id=0, realtime=False):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dosya bulunamadi: {path}")
        self.cap = cv2.VideoCapture(path)
        if not self.cap.isOpened():
            raise IOError(f"Video acilamadi (codec?): {path}")
        self.cam_id, self.realtime, self.index = cam_id, realtime, -1
        self.fps = self.cap.get(cv2.CAP_PROP_FPS) or 30.0
        self.width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self._t0 = time.monotonic()

    def read(self):
        ok, img = self.cap.read()
        if not ok: return None
        self.index += 1
        ts = self.index / self.fps
        if self.realtime:
            d = (self._t0 + ts) - time.monotonic()
            if d > 0: time.sleep(d)
        return Frame(img, ts, self.index, self.cam_id)

    def release(self): self.cap.release()


class CsiCameraSource(FrameSource):
    """Jetson CSI camera. Won't run off-device; here so deployment is a no-op."""
    GST = ("nvarguscamerasrc sensor-id={sid} ! "
           "video/x-raw(memory:NVMM), width={w}, height={h}, framerate={fps}/1 ! "
           "nvvidconv ! video/x-raw, format=BGRx ! "
           "videoconvert ! video/x-raw, format=BGR ! "
           "appsink drop=true max-buffers=1 sync=false")

    def __init__(self, sensor_id=0, width=1920, height=1080, fps=30, cam_id=None):
        self.cap = cv2.VideoCapture(
            self.GST.format(sid=sensor_id, w=width, h=height, fps=fps),
            cv2.CAP_GSTREAMER)
        if not self.cap.isOpened():
            raise IOError(f"CSI kamera acilamadi (sensor-id={sensor_id})")
        self.cam_id = sensor_id if cam_id is None else cam_id
        self.index = -1

    def read(self):
        ok, img = self.cap.read()
        if not ok: return None
        self.index += 1
        return Frame(img, time.monotonic(), self.index, self.cam_id)

    def release(self): self.cap.release()


class OnlineStabilizer:
    """
    frame -> downscale + gray -> LK tracking -> partial affine (RANSAC)
          -> cumulative trajectory -> EMA -> correction -> limit -> one warp
    """
    def __init__(self, cfg: StabConfig = StabConfig(), cam_id: int = 0):
        self.cfg, self.cam_id = cfg, cam_id
        if cfg.num_threads > 0: cv2.setNumThreads(cfg.num_threads)
        cv2.setUseOptimized(True)
        self._lk = dict(winSize=(cfg.lk_win, cfg.lk_win), maxLevel=cfg.lk_levels,
                        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 20, 0.03))
        self.reset()

    def reset(self):
        self.prev_gray = None
        self.prev_pts = None
        self.traj = np.zeros(3)
        self.smooth = np.zeros(3)
        self.frame_count = 0
        self.scale = 1.0
        self.size = None
        self._center = (0.0, 0.0)
        self._last_m = np.zeros(3)
        self._coast = 0
        self._zoom = 1.0 / (1.0 - 2.0 * self.cfg.crop_ratio)

    def _prepare(self, image):
        h, w = image.shape[:2]
        if self.size is None:
            self.size = (w, h)
            self.scale = min(1.0, self.cfg.proc_width / float(w))
            self._center = (w / 2.0, h / 2.0)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        if self.scale < 1.0:
            gray = cv2.resize(gray, None, fx=self.scale, fy=self.scale,
                              interpolation=cv2.INTER_AREA)
        return gray

    def _detect(self, gray, keep=None):
        """Detect features with a per-cell quota.

        goodFeaturesToTrack ranks corners globally, so one high-texture
        region -- a passing car, a close-up object -- can take most of the
        budget and drag RANSAC onto *its* motion instead of the camera's.
        Passing `keep` preserves surviving tracks and only fills the gaps."""
        cfg = self.cfg
        h, w = gray.shape
        gy = gx = cfg.grid
        per = max(4, cfg.max_corners // (gx*gy))
        mask = None
        if keep is not None and len(keep):
            mask = np.full((h, w), 255, np.uint8)
            for x, y in keep.reshape(-1, 2).astype(int):
                cv2.circle(mask, (x, y), cfg.min_distance, 0, -1)
        pts = []
        for i in range(gy):
            for j in range(gx):
                y0, y1 = i*h//gy, (i+1)*h//gy
                x0, x1 = j*w//gx, (j+1)*w//gx
                sub = None if mask is None else mask[y0:y1, x0:x1]
                p = cv2.goodFeaturesToTrack(gray[y0:y1, x0:x1], per,
                        cfg.quality_level, cfg.min_distance,
                        mask=sub, blockSize=cfg.block_size)
                if p is not None:
                    p[:,0,0] += x0; p[:,0,1] += y0
                    pts.append(p)
        new = np.vstack(pts) if pts else None
        if keep is None or not len(keep):
            return new
        return keep if new is None else np.vstack([keep, new]).astype(np.float32)

    def _track(self, pg, cg, p0):
        p1, st, _ = cv2.calcOpticalFlowPyrLK(pg, cg, p0, None, **self._lk)
        if p1 is None: return None, None
        st = st.reshape(-1).astype(bool)
        if self.cfg.use_fb_check and st.any():
            pb, st2, _ = cv2.calcOpticalFlowPyrLK(cg, pg, p1, None, **self._lk)
            if pb is not None:
                err = np.linalg.norm(p0.reshape(-1, 2) - pb.reshape(-1, 2), axis=1)
                st &= st2.reshape(-1).astype(bool) & (err < self.cfg.fb_threshold)
        if st.sum() < 6: return None, None
        return p0[st], p1[st]

    def _estimate(self, p0, p1) -> Motion:
        M, inl = cv2.estimateAffinePartial2D(
            p0, p1, method=cv2.RANSAC, ransacReprojThreshold=self.cfg.ransac_thresh,
            maxIters=500, confidence=0.99, refineIters=10)
        if M is None: return Motion()
        n = int(inl.sum()) if inl is not None else 0
        if n < self.cfg.min_inliers: return Motion(n_inliers=n)
        s = 1.0 / self.scale
        return Motion(float(M[0, 2]) * s, float(M[1, 2]) * s,
                      float(math.atan2(M[1, 0], M[0, 0])), True, n)

    def _fits(self, M, W, H):
        """Do all four output corners land inside the source frame?

        Checks rotation, zoom and translation together. Limiting each axis
        separately missed the extra margin rotation eats at the corners --
        at crop=0.10 and 6 degrees the frame overflowed by ~53 px and
        BORDER_REPLICATE smeared the edges."""
        Mi = cv2.invertAffineTransform(M)
        dst = np.array([[0,0],[W,0],[W,H],[0,H]], np.float32).reshape(-1,1,2)
        q = cv2.transform(dst, Mi).reshape(-1,2)
        return (q[:,0].min() >= 0 and q[:,1].min() >= 0
                and q[:,0].max() <= W and q[:,1].max() <= H)

    def _soft_limit(self, corr, mx, my, ma, knee=0.6):
        """Squash toward the limit with tanh past a knee instead of clipping.

        np.clip has a discontinuous derivative, so every time the correction
        hit the wall its velocity dropped to zero and the picture snapped --
        the most visible artifact this stabilizer produced."""
        lim = (mx, my, ma)
        for i in range(3):
            r = abs(corr[i]) / max(lim[i], 1e-9)
            if r > knee:
                t = (r - knee) / (1 - knee)
                corr[i] *= (knee + (1 - knee) * math.tanh(t)) / r
        return corr

    def _smooth_step(self, mx, my):
        cfg = self.cfg
        a = cfg.smooth_alpha
        self.smooth = a * self.smooth + (1 - a) * self.traj
        corr = self.smooth - self.traj
        if cfg.adaptive:
            r = max(abs(corr[0]) / max(mx, 1e-6), abs(corr[1]) / max(my, 1e-6))
            if r > 0.7:   # track the camera faster as we approach the limit
                ae = a - (a - 0.60) * min((r - 0.7) / 0.3, 1.0)
                self.smooth = ae * self.smooth + (1 - ae) * self.traj
                corr = self.smooth - self.traj
        ma = math.radians(cfg.max_angle_deg)
        corr = self._soft_limit(corr, mx, my, ma)
        # anti-windup: leak toward the limited value instead of snapping to it
        self.smooth += cfg.windup_leak * ((self.traj + corr) - self.smooth)
        return corr

    def _warp_matrix(self, corr):
        # Sign: estimateAffinePartial2D and getRotationMatrix2D disagree on
        # rotation direction, hence the minus. Without it the correction
        # roughly doubled the rotation instead of removing it.
        M = cv2.getRotationMatrix2D(self._center, -math.degrees(corr[2]), self._zoom)
        # The crop-zoom magnifies the image by z, so it magnifies the motion
        # inside it too. Without scaling the correction by z, a fraction
        # (z-1)/z of the shake survives -- 32% at crop=0.12.
        M[0, 2] += corr[0] * self._zoom
        M[1, 2] += corr[1] * self._zoom
        return M

    def process(self, frame: Frame, external_correction=None) -> StabilizedFrame:
        t0 = time.perf_counter()
        cfg = self.cfg
        gray = self._prepare(frame.image)
        W, H = self.size
        mx = cfg.crop_ratio * W * cfg.safety
        my = cfg.crop_ratio * H * cfg.safety

        motion = Motion()
        need = (self.prev_pts is None or len(self.prev_pts) < cfg.min_tracks
                or self.frame_count % cfg.redetect_interval == 0)
        if self.prev_gray is not None:
            if need: self.prev_pts = self._detect(self.prev_gray, keep=self.prev_pts)
            if self.prev_pts is not None and len(self.prev_pts) >= 6:
                p0, p1 = self._track(self.prev_gray, gray, self.prev_pts)
                if p0 is not None:
                    motion = self._estimate(p0, p1)
                    self.prev_pts = p1.reshape(-1, 1, 2).astype(np.float32)
                else:
                    self.prev_pts = None
            else:
                self.prev_pts = None

        if motion.valid:
    # Weight by inlier count rather than treating motion as all-or-nothing.
            # Assuming "the camera stopped" on a blurry frame made the
            # trajectory jump once tracking recovered.
            # Blend toward a constant-velocity prediction, not toward zero.
            # `traj += w * motion` looks like a confidence filter but is a
            # truncated estimate: on a frame with half the required inliers,
            # half the measured motion never reaches the trajectory. That error
            # is permanent -- traj drifts below the true camera path while the
            # warp is still applied to the real image. Measured on a synthetic
            # case: 100 px of true motion accumulated as 62 px.
            w = min(1.0, motion.n_inliers / float(cfg.conf_inliers))
            m = motion.as_array()
            est = w * m + (1.0 - w) * self._last_m
            self.traj += est
            self._last_m = 0.7 * est + 0.3 * self._last_m
            self._coast = 3
        elif self._coast > 0:
            # tracking lost: coast on last known velocity for a few frames
            # rather than asserting the camera stopped
            self.traj += self._last_m
            self._coast -= 1
        corr = self._smooth_step(mx, my)
        if external_correction is not None:
            corr = np.asarray(external_correction, dtype=np.float64)
            self.smooth = self.traj + corr

        M = self._warp_matrix(corr)
        for _ in range(6):                      # shrink until it fits
            if self._fits(M, W, H): break
            corr *= 0.85
            M = self._warp_matrix(corr)
        out = cv2.warpAffine(frame.image, M, (W, H),
                             flags=cfg.interpolation, borderMode=cfg.border_mode)
        self.prev_gray = gray
        self.frame_count += 1
        return StabilizedFrame(out, frame.timestamp, frame.index, frame.cam_id,
                               corr, motion, M,
                               0 if self.prev_pts is None else len(self.prev_pts),
                               (time.perf_counter() - t0) * 1000.0)


class CameraPipeline:
    """One source + one stabilizer. The sync layer will hold two of these."""
    def __init__(self, source, cfg=StabConfig(), cam_id=0):
        self.source, self.cam_id = source, cam_id
        self.stab = OnlineStabilizer(cfg, cam_id=cam_id)
    def next(self, external_correction=None):
        f = self.source.read()
        if f is None: return None
        f.cam_id = self.cam_id
        return self.stab.process(f, external_correction=external_correction)
    def release(self): self.source.release()


print("Core loaded.")

### A2. Measurement and benchmark helpers


In [ ]:
"""
Offline evaluation helpers. None of this ships to the Jetson -- it exists to
measure whether the stabilizer is doing its job and to pick parameters.

Two metrics live here and they answer different questions:

  measure_motion / compare_motion
      Mean frame-to-frame displacement of the output video. Includes the
      deliberate camera motion we are supposed to *keep*, so on pan-heavy
      footage it can never approach 100%. Useful, but easy to misread.

  high-frequency suppression (see hf_score)
      Removes the low-frequency component first, then compares what is left.
      This is the one that tracks what the eye calls "shaky", and it is the
      metric parameter choices should be made against.
"""

import math
import os
import subprocess
from typing import Optional
from urllib.request import urlretrieve

import cv2
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


SAMPLES = {
    # MeshFlow (Liu et al., ECCV 2016) sample clip -- academic source, fast host
    "meshflow": ("https://raw.githubusercontent.com/sudheerachary/"
                 "Mesh-Flow-Video-Stabilization/master/data/shaky-5.avi"),
    "meshflow_small": ("https://raw.githubusercontent.com/sudheerachary/"
                       "Mesh-Flow-Video-Stabilization/master/data/small-shaky-5.avi"),
    "ostrich": "https://s3.amazonaws.com/python-vidstab/ostrich.mp4",
    "thrasher": "https://s3.amazonaws.com/python-vidstab/thrasher.mp4",
}


def download_sample(name="meshflow", dest="samples"):
    if name not in SAMPLES:
        raise ValueError(f"Unknown sample: {name}. Options: {list(SAMPLES)}")
    os.makedirs(dest, exist_ok=True)
    ext = os.path.splitext(SAMPLES[name])[1] or ".mp4"
    path = os.path.join(dest, f"{name}{ext}")
    if os.path.exists(path) and os.path.getsize(path) > 100_000:
        print(f"[cached] {path}")
        return path
    print(f"[downloading] {SAMPLES[name]}")
    urlretrieve(SAMPLES[name], path)
    c = cv2.VideoCapture(path)
    print(f"[ready] {path}  {int(c.get(3))}x{int(c.get(4))} "
          f"{c.get(5):.1f}fps {int(c.get(7))} frames "
          f"({os.path.getsize(path)/1e6:.1f} MB)")
    c.release()
    return path


def add_shake(src_path, out_path=None, amp=15.0, rot=2.0, noise=0.4,
              max_frames=300, seed=0):
    """Inject shake of known strength and save the ground truth alongside.

    The ground truth is stored as the affine parameters of the injected
    matrix, not as the raw jx/jy offsets: rotation about the centre produces
    extra translation away from the centre, and the estimator measures that
    inside tx/ty. Storing jx/jy would mean the two sides measure different
    things and the suppression numbers come out nonsensical.
    """
    rng = np.random.default_rng(seed)
    if out_path is None:
        b, e = os.path.splitext(src_path)
        out_path = f"{b}_shake_a{amp:g}_r{rot:g}{e}"   # params in the filename
    if os.path.exists(out_path) and os.path.exists(out_path + ".gt.npy"):
        print(f"[cached] {out_path}")
        return out_path, out_path + ".gt.npy"

    src = VideoFileSource(src_path)
    W, H = src.width, src.height
    w = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), src.fps, (W, H))
    gt, i = [], 0
    while True:
        f = src.read()
        if f is None or (max_frames and i >= max_frames):
            break
        jx = amp * math.sin(i*1.7) + amp*noise*rng.standard_normal()
        jy = amp*0.8*math.sin(i*2.3) + amp*noise*rng.standard_normal()
        ja = rot * math.sin(i*1.1) + rot*noise*rng.standard_normal()
        M = cv2.getRotationMatrix2D((W/2., H/2.), ja, 1.0)
        M[0, 2] += jx
        M[1, 2] += jy
        w.write(cv2.warpAffine(f.image, M, (W, H), borderMode=cv2.BORDER_REPLICATE))
        gt.append([float(M[0, 2]), float(M[1, 2]),
                   float(math.atan2(M[1, 0], M[0, 0]))])
        i += 1
    w.release()
    src.release()
    np.save(out_path + ".gt.npy", np.array(gt))
    print(f"[generated] {out_path}  ({i} frames, amp={amp}, rot={rot})")
    return out_path, out_path + ".gt.npy"


def stabilize_clip(path, cfg=None, out_dir="out", max_frames=None,
                   side_by_side=True, to_h264=True, quiet=False):
    """Run a clip through the stabilizer one frame at a time, as if live."""
    cfg = cfg or StabConfig()
    os.makedirs(out_dir, exist_ok=True)
    name = os.path.splitext(os.path.basename(path))[0]
    tmp = os.path.join(out_dir, f"{name}_tmp.mp4")
    final = os.path.join(out_dir, f"{name}_cmp.mp4" if side_by_side
                         else f"{name}_stab.mp4")
    src = VideoFileSource(path)
    stab = OnlineStabilizer(cfg)
    W, H = src.width, src.height
    wr = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*"mp4v"), src.fps,
                         (W*2 if side_by_side else W, H))
    times, raw_j, stab_j, corrs = [], [], [], []
    prev = np.zeros(3)
    n = 0
    while True:
        f = src.read()
        if f is None:
            break
        sf = stab.process(f)
        times.append(sf.proc_ms)
        corrs.append(sf.correction.copy())
        if sf.motion.valid:
            raw_j.append(math.hypot(sf.motion.dx, sf.motion.dy))
            z = stab._zoom            # the zoom magnifies output motion by z
            r = z * (sf.motion.as_array() + (sf.correction - prev))
            stab_j.append(math.hypot(r[0], r[1]))
        prev = sf.correction.copy()
        wr.write(np.hstack([f.image, sf.image]) if side_by_side else sf.image)
        n += 1
        if max_frames and n >= max_frames:
            break
    wr.release()
    src.release()

    if to_h264 and subprocess.call(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp,
                                    "-vcodec", "libx264", "-pix_fmt", "yuv420p",
                                    "-crf", "24", final]) == 0:
        os.remove(tmp)
    else:
        os.replace(tmp, final)

    t = np.array(times) if times else np.array([0.0])
    res = {"clip": os.path.basename(path), "res": f"{W}x{H}", "frames": n,
           "ms": float(t.mean()), "ms_p95": float(np.percentile(t, 95)),
           "fps": float(1000/t.mean()) if t.mean() else 0.0,
           "jit_raw": float(np.mean(raw_j)) if raw_j else 0.0,
           "jit_stab": float(np.mean(stab_j)) if stab_j else 0.0,
           "corrections": np.array(corrs), "times": t, "cfg": cfg, "out": final}
    if not quiet:
        red = (1 - res["jit_stab"]/res["jit_raw"])*100 if res["jit_raw"] else 0
        print(f"{res['res']} | {n} frames | {res['ms']:.2f} ms "
              f"(p95 {res['ms_p95']:.2f}) -> {res['fps']:.1f} FPS")
        print(f"jitter: {res['jit_raw']:.2f} -> {res['jit_stab']:.2f} px/frame "
              f"({red:.1f}% reduction)")
    return res


# --------------------------------------------------------------------------
# metrics
# --------------------------------------------------------------------------

def measure_motion(path, max_frames=400, half=None):
    """Frame-to-frame motion of a video on its own.

    half='right' measures only the stabilized half of a side-by-side output;
    without it half the frame is the raw footage and the number is garbage.
    """
    stab = OnlineStabilizer(StabConfig(crop_ratio=0.0, smooth_alpha=0.0))
    src = VideoFileSource(path)
    m, i = [], 0
    while i < max_frames:
        f = src.read()
        if f is None:
            break
        if half:
            w = f.image.shape[1] // 2
            f.image = f.image[:, :w] if half == "left" else f.image[:, w:]
        sf = stab.process(f)
        if sf.motion.valid:
            m.append(sf.motion.as_array())
        i += 1
    src.release()
    if not m:
        return {"dxy": 0.0, "da_std": 0.0, "frames": 0}
    m = np.array(m)
    return {"dxy": float(np.mean(np.hypot(m[:, 0], m[:, 1]))),
            "da_std": float(np.degrees(np.std(m[:, 2]))), "frames": len(m)}


def compare_motion(in_path, out_path, max_frames=400, out_half=None, out_zoom=1.0):
    """Input vs output motion.

    out_zoom must be 1/(1-2*crop_ratio). The output is zoomed, so everything
    in it -- including the camera motion we deliberately keep -- is magnified
    by z. Skip the division and results get worse as crop_ratio goes up, which
    is an artifact, not a real effect. Rotation is unaffected by zoom.
    """
    a = measure_motion(in_path, max_frames)
    b = measure_motion(out_path, max_frames, half=out_half)
    b = {**b, "dxy": b["dxy"] / out_zoom}
    dr = (1 - b["dxy"]/a["dxy"])*100 if a["dxy"] else 0.0
    ar = (1 - b["da_std"]/a["da_std"])*100 if a["da_std"] else 0.0
    print(f"{'':8s} {'translation':>13s} {'rotation':>12s}")
    print(f"{'input':8s} {a['dxy']:10.2f} px {a['da_std']:9.2f} deg")
    print(f"{'output':8s} {b['dxy']:10.2f} px {b['da_std']:9.2f} deg")
    print(f"{'reduced':8s} {dr:10.1f} %  {ar:9.1f} %")
    return {"in": a, "out": b, "dxy_red": dr, "da_red": ar}


def _trajectory(path, n=300):
    st = OnlineStabilizer(StabConfig(crop_ratio=0.0, smooth_alpha=0.0))
    src = VideoFileSource(path)
    m, i = [], 0
    while i < n:
        f = src.read()
        if f is None:
            break
        sf = st.process(f)
        if sf.motion.valid:
            m.append(sf.motion.as_array())
        i += 1
    src.release()
    return np.cumsum(np.array(m), axis=0)


def _highpass(x, w=15):
    pad = np.pad(x, ((w//2, w//2), (0, 0)), mode="edge")
    return x - sliding_window_view(pad, w, axis=0).mean(axis=-1)


def hf_score(path, cfg, n=300, out_dir="hf"):
    """High-frequency suppression: the metric that matches what the eye sees.

    Strips the low-frequency component (the deliberate camera motion) from
    both input and output, then compares the standard deviation of what is
    left. Returns (x %, y %, saturation %).
    """
    r = stabilize_clip(path, cfg, out_dir=out_dir, max_frames=n,
                       side_by_side=False, to_h264=False, quiet=True)
    z = 1/(1-2*cfg.crop_ratio)
    a = _highpass(_trajectory(path, n))
    b = _highpass(_trajectory(r["out"], n)) / z
    c = cv2.VideoCapture(path)
    W, H = int(c.get(3)), int(c.get(4))
    c.release()
    mx, my = cfg.crop_ratio*W*0.95, cfg.crop_ratio*H*0.95
    co = r["corrections"]
    sat = max((np.abs(co[:, 0]) >= mx*.98).mean(),
              (np.abs(co[:, 1]) >= my*.98).mean())
    return (100*(1-np.std(b[:, 0])/np.std(a[:, 0])),
            100*(1-np.std(b[:, 1])/np.std(a[:, 1])), 100*sat)


def evaluate_gt(gt_path, result, baseline=None, verbose=True):
    """How much of the injected shake came back out.

    `baseline` is the same clip stabilized *without* injected shake. Subtracting
    it removes the source footage's own shake, which otherwise counts as error.
    Note this assumes the system is linear, so it stops being valid once the
    correction saturates -- a clip whose deliberate motion already fills the
    crop budget will produce meaningless (sometimes negative) numbers here.
    """
    gt = np.load(gt_path)
    corr = result["corrections"]
    n = min(len(gt), len(corr))
    gt, corr = gt[:n], corr[:n]
    resid = gt + corr
    if baseline is not None:
        b = baseline["corrections"]
        m = min(n, len(b))
        gt, corr, resid, n = gt[:m], corr[:m], resid[:m] - b[:m], m
    before, after = np.std(gt[:, :2], axis=0), np.std(resid[:, :2], axis=0)
    ba, aa = math.degrees(np.std(gt[:, 2])), math.degrees(np.std(resid[:, 2]))
    out = {"n": n, "baseline": baseline is not None,
           "x_before": float(before[0]), "x_after": float(after[0]),
           "y_before": float(before[1]), "y_after": float(after[1]),
           "a_before": ba, "a_after": aa,
           "x_red": float((1-after[0]/before[0])*100) if before[0] else 0.0,
           "y_red": float((1-after[1]/before[1])*100) if before[1] else 0.0,
           "a_red": float((1-aa/ba)*100) if ba else 0.0,
           "gt": gt, "resid": resid, "corr": corr}
    if verbose:
        tag = "baseline subtracted" if baseline is not None else "no baseline"
        print(f"ground truth, {n} frames, {tag}")
        print(f"{'axis':>6s} {'injected':>10s} {'residual':>10s} {'suppressed':>11s}")
        print(f"{'x':>6s} {out['x_before']:9.2f}px {out['x_after']:9.2f}px {out['x_red']:10.1f}%")
        print(f"{'y':>6s} {out['y_before']:9.2f}px {out['y_after']:9.2f}px {out['y_red']:10.1f}%")
        print(f"{'angle':>6s} {out['a_before']:9.2f}d  {out['a_after']:9.2f}d  {out['a_red']:10.1f}%")
    return out


def show(path, width=960):
    """Play a clip inline. Left half raw, right half stabilized."""
    from base64 import b64encode
    from IPython.display import HTML
    mb = os.path.getsize(path)/1e6
    if mb > 60:
        print(f"warning: {mb:.0f} MB embedded; lower max_frames")
    d = b64encode(open(path, "rb").read()).decode()
    return HTML(f'<video width={width} controls loop>'
                f'<source src="data:video/mp4;base64,{d}" type="video/mp4"></video>')


print("Tools loaded.")

### A3. Plots and batch runner


In [ ]:
"""Plotting helpers for the notebook."""

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt


def summary_table(results):
    print(f"{'clip':12s} {'resolution':>11s} {'frames':>7s} {'ms':>7s} {'p95':>7s} "
          f"{'FPS':>6s} {'transl.':>9s} {'rot.':>8s} {'sat_x':>7s} {'sat_y':>7s}")
    print("-" * 92)
    for r in results.values():
        rot = "n/a" if r.get("rot_valid") is False else f"{r['da_red']:7.1f}%"
        print(f"{r['name']:12s} {r['res']:>11s} {r['frames']:7d} {r['ms']:7.2f} "
              f"{r['ms_p95']:7.2f} {r['fps']:6.1f} {r['dxy_red']:8.1f}% "
              f"{rot:>8s} {r['sat_x']:6.1f}% {r['sat_y']:6.1f}%")
    print("-" * 92)
    print("transl./rot. = end-to-end suppression, zoom corrected")
    print("sat_*        = share of frames where the correction hit the crop limit")
    print("rot. = n/a   = clip has < 0.2 deg of rotation; the ratio is noise")


def plot_all(results, cfg):
    names = list(results)
    idx = np.arange(len(names))
    w = .38
    fig, ax = plt.subplots(2, 2, figsize=(15, 9))
    fig.suptitle(f"alpha={cfg.smooth_alpha}, crop={cfg.crop_ratio}, "
                 f"adaptive={cfg.adaptive}", fontsize=13, fontweight="bold")

    a = ax[0, 0]
    b1 = a.bar(idx-w/2, [results[n]["dxy_red"] for n in names], w,
               label="translation", color="royalblue", alpha=.85)
    b2 = a.bar(idx+w/2, [results[n]["da_red"] for n in names], w,
               label="rotation", color="darkorange", alpha=.85)
    for bs in (b1, b2):
        for bar in bs:
            a.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
                   f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=8)
    a.set_title("End-to-end suppression"); a.set_ylabel("%")
    a.set_xticks(idx); a.set_xticklabels(names); a.legend(fontsize=8)
    a.grid(alpha=.3, axis="y")

    a = ax[0, 1]
    a.bar(idx-w/2, [results[n]["sat_x"] for n in names], w, label="x",
          color="royalblue", alpha=.85)
    a.bar(idx+w/2, [results[n]["sat_y"] for n in names], w, label="y",
          color="seagreen", alpha=.85)
    a.axhline(10, ls=":", c="red", lw=1.3, label="10% threshold")
    a.set_title("Frames at the crop limit"); a.set_ylabel("% of frames")
    a.set_xticks(idx); a.set_xticklabels(names); a.legend(fontsize=8)
    a.grid(alpha=.3, axis="y")

    a = ax[1, 0]
    for n in names:
        t = results[n]["times"]
        a.plot(np.arange(len(t)), t, lw=.8, alpha=.75, label=n)
    a.axhline(33.33, ls=":", c="red", lw=1.4, label="30 FPS budget")
    a.set_title("Per-frame processing time")
    a.set_xlabel("frame"); a.set_ylabel("ms"); a.legend(fontsize=8); a.grid(alpha=.3)

    a = ax[1, 1]
    for n in names:
        co = results[n]["corrections"]
        a.plot(np.hypot(co[:, 0], co[:, 1]), lw=1.0, alpha=.8, label=n)
    a.set_title("Magnitude of the applied correction")
    a.set_xlabel("frame"); a.set_ylabel("pixels"); a.legend(fontsize=8); a.grid(alpha=.3)

    plt.tight_layout(); plt.show()


def plot_trajectory(path, cfg, max_frames=300):
    """Raw vs smoothed trajectory. The shaded gap is the correction; the dashed
    red lines are the crop limit. If the correction rides the limit, crop_ratio
    is too small for that clip."""
    src = VideoFileSource(path)
    stab = OnlineStabilizer(cfg)
    traj, sm, n = [], [], 0
    while n < max_frames:
        f = src.read()
        if f is None:
            break
        stab.process(f)
        traj.append(stab.traj.copy())
        sm.append(stab.smooth.copy())
        n += 1
    src.release()
    traj, sm = np.array(traj), np.array(sm)
    W, H = stab.size
    lim = [cfg.crop_ratio*W*0.95, cfg.crop_ratio*H*0.95]
    labels = ["x (px)", "y (px)", "angle (deg)"]
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    for i, a in enumerate(axes):
        t = np.degrees(traj[:, i]) if i == 2 else traj[:, i]
        s = np.degrees(sm[:, i]) if i == 2 else sm[:, i]
        a.plot(t, lw=1.0, alpha=.75, label="raw trajectory")
        a.plot(s, lw=1.8, label="smoothed (applied)")
        a.fill_between(range(len(t)), t, s, alpha=.18, label="correction")
        if i < 2:
            a.plot(t+lim[i], "--", lw=.7, c="red", alpha=.5)
            a.plot(t-lim[i], "--", lw=.7, c="red", alpha=.5, label="crop limit")
        a.set_ylabel(labels[i]); a.legend(fontsize=8, loc="upper right"); a.grid(alpha=.3)
    axes[-1].set_xlabel("frame")
    fig.suptitle(f"{os.path.basename(path)} — alpha={cfg.smooth_alpha}, "
                 f"crop={cfg.crop_ratio}")
    plt.tight_layout(); plt.show()


print("Plots loaded.")


"""Batch runner: same config across every clip so comparisons are fair."""

import cv2
import numpy as np


ROT_MIN_DEG = 0.2      # below this the rotation ratio is noise, not signal


def run_all(clips, cfg, max_frames=None):
    z = 1 / (1 - 2*cfg.crop_ratio)
    results = {}
    for name, path in clips.items():
        print(f"[{name}] running...", flush=True)
        r = stabilize_clip(path, cfg, out_dir="out", max_frames=max_frames,
                           side_by_side=True, quiet=True)
        n = r["frames"]
        a = measure_motion(path, n)
        b = measure_motion(r["out"], n, half="right")
        b = {**b, "dxy": b["dxy"] / z}          # output is zoomed
        c = cv2.VideoCapture(path)
        W, H = int(c.get(3)), int(c.get(4))
        c.release()
        mx, my = cfg.crop_ratio*W*0.95, cfg.crop_ratio*H*0.95
        co = r["corrections"]
        r.update({
            "name": name, "in_path": path, "zoom": z, "m_in": a, "m_out": b,
            "dxy_red": (1 - b["dxy"]/a["dxy"])*100 if a["dxy"] else 0.0,
            "da_red": (1 - b["da_std"]/a["da_std"])*100 if a["da_std"] else 0.0,
            "rot_valid": a["da_std"] >= ROT_MIN_DEG,
            "sat_x": float((np.abs(co[:, 0]) >= mx*0.98).mean()*100),
            "sat_y": float((np.abs(co[:, 1]) >= my*0.98).mean()*100),
        })
        results[name] = r
    print("done.")
    return results


print("Runner loaded.")

---
## B. Data and settings

`meshflow` and `ostrich` download automatically. `running.mp4` and any Jetson
recordings you want to test go in the file panel on the left.


In [ ]:
import os

CLIPS = {}
CLIPS["meshflow"] = download_sample("meshflow")   # MeshFlow ECCV 2016 sample
CLIPS["ostrich"] = download_sample("ostrich")     # handheld

for extra in ("running.mp4",):                    # drop your own clips here
    if os.path.exists(extra):
        CLIPS[os.path.splitext(extra)[0]] = extra
    else:
        print(f"note: {extra} not found, skipping")

print()
for n, p in CLIPS.items():
    c = cv2.VideoCapture(p)
    print(f"{n:12s} {int(c.get(3))}x{int(c.get(4))}  {c.get(5):4.1f}fps  "
          f"{int(c.get(7)):5d} frames")
    c.release()

### Settings

One config for every clip, otherwise the comparison isn't fair.

These values come from the sweeps in the appendix. `crop_ratio=0.08` sits at the
knee of the crop-vs-suppression curve: going to 0.10 costs another 4% of the
frame and buys about 2 points. `adaptive` is off because it measured worse
almost everywhere.


In [ ]:
CFG = StabConfig(
    smooth_alpha=0.92,
    crop_ratio=0.08,       # 16% of the frame, zoom 1.19x
    adaptive=False,
    max_angle_deg=3.0,
)
MAX_FRAMES = 300

print(f"alpha={CFG.smooth_alpha}  crop={CFG.crop_ratio}  "
      f"zoom={1/(1-2*CFG.crop_ratio):.3f}  adaptive={CFG.adaptive}")

---
## C. Run


In [ ]:
RESULTS = run_all(CLIPS, CFG, MAX_FRAMES)

---
## D. Results

Two things to watch. `sat_x`/`sat_y` above 10% means the correction is hitting
the crop limit on that clip and `crop_ratio` is the binding constraint, not the
filter. Rotation shows `n/a` when a clip has almost no rotation to begin with --
dividing a small number by a small number just measures noise.


In [ ]:
summary_table(RESULTS)

### Charts


In [ ]:
plot_all(RESULTS, CFG)

### Videos

Left half raw, right half stabilized. The numbers can only take you so far;
this is where the decision actually gets made.


In [ ]:
for n, r in RESULTS.items():
    print(f"\n=== {n}  ({r['res']}, translation -{r['dxy_red']:.0f}%, "
          f"rotation -{r['da_red']:.0f}%) ===")
    display(show(r["out"]))

### One clip in detail


In [ ]:
CLIP = "meshflow"
plot_trajectory(CLIPS[CLIP], CFG, MAX_FRAMES)

---
## E. Deployment

The Jetson only needs section A1. Copy it to `core.py`, pair it with
`run_camera.py` from the repo, and swap `VideoFileSource` for
`CsiCameraSource`.

```bash
gst-launch-1.0 nvarguscamerasrc num-buffers=30 ! fakesink   # camera alive?
python3 record_clips.py --sensor-id 0 --seconds 30          # get real footage
python3 run_camera.py --sensor-id 0 --frames 900 --no-display
```

Measure p95, not the mean: 30 FPS means every frame under 33.3 ms, and an
average of 20 ms with a p95 of 40 ms still drops frames. Close VS Code first,
its server eats a real share of the CPU on an Orin Nano.

If p95 goes over budget, in order: `--proc-width 320`, then `--cuda` for the
warp, then VPI for optical flow.


---
# Appendix — how the settings were chosen

Nothing below runs as part of the main flow. These are the experiments that
produced the defaults, kept because the reasoning matters more than the
numbers and because a few of them found real bugs.

Each section says what it measures and what came out of it.


## A. Which metric to trust

End-to-end suppression counts the deliberate camera motion we're supposed to
keep, so on pan-heavy footage it can't get near 100%. Splitting the motion into
what's intentional and what's shake makes that concrete: on one of the Jetson
recordings 97% of the movement was intentional, which is why 40% looked like a
bad score when it was close to the ceiling.

`hf_score` strips the low-frequency part first. That's the number worth
optimizing, and switching to it changed several conclusions.


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def motion_split(path, n=300):
    """Split the trajectory into intentional motion and shake."""
    cum = _trajectory(path, n)
    def lowpass(x, w=31):
        pad = np.pad(x, ((w//2, w//2), (0, 0)), mode="edge")
        return sliding_window_view(pad, w, axis=0).mean(axis=-1)
    lp = lowpass(cum); hp = cum - lp
    print(f"{os.path.basename(path)}")
    for i, lbl in enumerate(["x", "y"]):
        low, high = np.std(lp[:, i]), np.std(hp[:, i])
        print(f"  {lbl}: intentional={low:7.1f}px  shake={high:6.1f}px"
              f"  -> shake is {100*high**2/(high**2+low**2):.0f}% of the energy")

for n, p in CLIPS.items():
    motion_split(p, 300)

## B. Crop ratio

Every pixel of crop is field of view given up, so this is the trade-off that
actually needs a decision. The curve has a knee around 0.08: below it the
suppression falls off quickly, above it each extra 4% of the frame buys about
2 points.

Worth showing to whoever owns the FOV budget rather than picking alone.


In [77]:
for cr in (0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.14):
    row = f"crop={cr:<5} zoom={1/(1-2*cr):.2f}x  loss={200*cr:3.0f}%  "
    for n in ("meshflow", "running"):
        if n not in CLIPS: continue
        x, y, s = hf_score(CLIPS[n], StabConfig(smooth_alpha=0.92, crop_ratio=cr,
                                                adaptive=False, max_angle_deg=3.0))
        row += f"| {n}: x={x:5.1f} y={y:5.1f} sat={s:3.0f}% "
    print(row)

crop=0.03  zoom=1.06x  loss=  6%  | meshflow: x= 38.5 y= 35.2 sat=  2% | running: x= 14.7 y= 14.4 sat=  4% 
crop=0.04  zoom=1.09x  loss=  8%  | meshflow: x= 49.4 y= 44.3 sat=  3% | running: x= 19.6 y= 18.0 sat=  6% 
crop=0.05  zoom=1.11x  loss= 10%  | meshflow: x= 56.5 y= 51.5 sat=  3% | running: x= 24.6 y= 21.7 sat=  8% 
crop=0.06  zoom=1.14x  loss= 12%  | meshflow: x= 62.1 y= 56.7 sat=  2% | running: x= 26.9 y= 25.8 sat=  7% 
crop=0.08  zoom=1.19x  loss= 16%  | meshflow: x= 70.0 y= 63.4 sat=  0% | running: x= 31.6 y= 31.4 sat=  6% 
crop=0.1   zoom=1.25x  loss= 20%  | meshflow: x= 72.5 y= 66.3 sat=  0% | running: x= 38.4 y= 34.2 sat=  4% 
crop=0.14  zoom=1.39x  loss= 28%  | meshflow: x= 74.2 y= 67.4 sat=  0% | running: x= 45.8 y= 40.9 sat=  2% 


## C. Smoothing strength

Two clips want opposite things. `meshflow` (walking) improves steadily as alpha
goes up. `running` gets worse — the camera moves fast enough that the extra lag
fills the crop budget and the correction saturates.

0.92 is the compromise. On footage with sustained fast motion 0.85 is the better
choice, which is a finding rather than a problem.


In [78]:
for al in (0.85, 0.92, 0.95):
    row = f"alpha={al}  "
    for n in ("meshflow", "running"):
        if n not in CLIPS: continue
        x, y, s = hf_score(CLIPS[n], StabConfig(smooth_alpha=al, crop_ratio=0.08,
                                                adaptive=False, max_angle_deg=3.0))
        row += f"| {n}: x={x:5.1f} y={y:5.1f} sat={s:3.0f}% "
    print(row)

KeyboardInterrupt: 

## D. Other smoothers

An EMA lags behind sustained motion by roughly `v * a/(1-a)`, and that lag eats
crop budget. Several filters that shouldn't have that problem were tried:
alpha-beta and alpha-beta-gamma (track constant velocity/acceleration with no
steady-state error), and cascaded EMAs (sharper cutoff).

None of them beat the plain EMA. The cascades push saturation up because the
extra stages add their own delay. A look-ahead variant was also tested offline
with up to 400 ms of buffering and still didn't win, which is worth reporting on
its own: the real-time constraint isn't costing anything here.


In [ ]:
def set_smoother(kind):
    """kind: 'ema' | 'ab' (constant velocity) | 'ema2' (cascaded)"""
    def _step(self, mx, my):
        cfg = self.cfg; al = cfg.smooth_alpha
        if kind == "ab":
            if not hasattr(self, "_vel"): self._vel = np.zeros(3)
            g = 1-al; bg = g*g/(2-g)
            pred = self.smooth + self._vel; res = self.traj - pred
            self.smooth = pred + g*res; self._vel = self._vel + bg*res
        elif kind == "ema2":
            if not hasattr(self, "_s1"): self._s1 = np.zeros(3)
            self._s1 = al*self._s1 + (1-al)*self.traj
            self.smooth = al*self.smooth + (1-al)*self._s1
        else:
            self.smooth = al*self.smooth + (1-al)*self.traj
        corr = self.smooth - self.traj
        ma = math.radians(cfg.max_angle_deg)
        corr = self._soft_limit(corr, mx, my, ma)
        self.smooth += cfg.windup_leak * ((self.traj + corr) - self.smooth)
        return corr
    OnlineStabilizer._smooth_step = _step

P = CLIPS["meshflow"]
for kind in ("ema", "ab", "ema2"):
    for al in (0.85, 0.92):
        set_smoother(kind)
        x, y, s = hf_score(P, StabConfig(smooth_alpha=al, crop_ratio=0.08,
                                         adaptive=False, max_angle_deg=3.0))
        print(f"{kind:5s} alpha={al}  x={x:5.1f}  y={y:5.1f}  sat={s:3.0f}%")
set_smoother("ema")   # back to default

## E. Ground truth

Inject shake of a known strength, then check how much came back out. This is the
cleanest signal that the core is correct — it's how the rotation sign bug and
the zoom scaling bug were both caught.

One caveat: subtracting the baseline assumes the system is linear, so it breaks
once the correction saturates. A clip whose own motion already fills the crop
budget will produce meaningless (sometimes negative) numbers here.


In [ ]:
P = CLIPS["meshflow"]
cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.10,
                 adaptive=False, max_angle_deg=6.0)

shaken, gt_path = add_shake(P, amp=15.0, rot=2.0, max_frames=300)
base = stabilize_clip(P, cfg, out_dir="gt_base", max_frames=300,
                      side_by_side=False, to_h264=False, quiet=True)
res = stabilize_clip(shaken, cfg, max_frames=300, quiet=True)
evaluate_gt(gt_path, res, baseline=base)

### Rotation on its own

Rotation suppression sits below translation, so it was isolated with a pure
rotation injection (`amp=0`). It came out around 73–74% on both clips, which
rules out a sign or composition error in the warp — it's just the ceiling.


In [ ]:
for clip in ("meshflow", "ostrich"):
    p = CLIPS[clip]
    sh, gtp = add_shake(p, amp=0.0, rot=3.0, max_frames=250)
    cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.10,
                     adaptive=False, max_angle_deg=6.0)
    base = stabilize_clip(p, cfg, out_dir=f"pr_{clip}", max_frames=250,
                          side_by_side=False, to_h264=False, quiet=True)
    r = stabilize_clip(sh, cfg, out_dir=f"pr2_{clip}", max_frames=250,
                       side_by_side=False, to_h264=False, quiet=True)
    ev = evaluate_gt(gtp, r, baseline=base, verbose=False)
    print(f"{clip:10s} pure rotation suppressed: {ev['a_red']:.1f}%")

## F. Where the remaining shake comes from

This one explains the ceiling. Split the frame into quadrants and measure each
one separately: if the quadrants disagree more than they agree, the leftover
motion isn't global and no single affine transform can remove it — fixing one
corner necessarily breaks another.

On the Jetson recordings the disagreement was larger than the common motion
(116% on input, 131% on output). That's rolling shutter, parallax, or both, and
it's the reason parameter tuning stopped helping. Getting past it means a
spatially-varying model such as MeshFlow, not a better filter.


In [ ]:
def quadrant_test(path, cfg, n=200):
    r = stabilize_clip(path, cfg, out_dir="quad", max_frames=n,
                       side_by_side=False, to_h264=False, quiet=True)
    for label, vid in [("input", path), ("output", r["out"])]:
        src = VideoFileSource(vid)
        sts = [OnlineStabilizer(StabConfig(crop_ratio=0.0, smooth_alpha=0.0))
               for _ in range(4)]
        acc = [[] for _ in range(4)]
        i = 0
        while i < n:
            f = src.read()
            if f is None: break
            h, w = f.image.shape[:2]
            quads = [(0,h//2,0,w//2), (0,h//2,w//2,w),
                     (h//2,h,0,w//2), (h//2,h,w//2,w)]
            for k, (y0,y1,x0,x1) in enumerate(quads):
                sub = Frame(f.image[y0:y1, x0:x1], f.timestamp, f.index)
                sf = sts[k].process(sub)
                if sf.motion.valid: acc[k].append(sf.motion.as_array()[:2])
            i += 1
        src.release()
        A = [np.array(a) for a in acc]
        m = min(len(a) for a in A)
        A = np.stack([a[:m] for a in A])
        common = A.mean(axis=0)
        spread = np.linalg.norm(A - common, axis=2).mean()
        mag = np.linalg.norm(common, axis=1).mean()
        print(f"{label:7s} common={mag:6.2f}px  spread={spread:6.2f}px  "
              f"ratio={100*spread/max(mag,1e-6):.0f}%")

quadrant_test(CLIPS["meshflow"], CFG)

## G. Bugs this process caught

Worth writing down, because each was found by a measurement rather than by
reading the code, and each moved the numbers.

**Rotation sign.** `estimateAffinePartial2D` and `getRotationMatrix2D` disagree
on which way positive rotation goes. Measuring the angle and applying it back
with the same sign roughly doubled the rotation. A/B test on a synthetic clip:
input 2.64°, output 4.93° before the fix, 0.67° after.

**Zoom scaling.** The crop-zoom magnifies the image by z, so it magnifies the
motion inside it too, but the correction wasn't scaled to match. A fraction
`(z-1)/z` of every shake survived — 32% at crop=0.12. Controlled test: 28.4 px
input, 10.3 px output before, 2.1 px after.

**Rotation budget.** Translation was limited per-axis, which ignores the extra
margin rotation eats at the corners. At crop=0.10 with 6° of rotation the frame
overflowed by 53 px and `BORDER_REPLICATE` smeared the edges — hard to name,
easy to see. Now all four corners are checked against the combined transform.

**Measurement bias.** `compare_motion` compared an unzoomed input against a
zoomed output, so results got *worse* as crop_ratio went up. Correcting for zoom
flattened it: what looked like 51%→29% across crop 0.06–0.20 was 57% throughout.

**Metric choice.** End-to-end motion rewards lag, which made a look-ahead
variant look worse than the causal one and pushed alpha too high. Switching to
high-frequency suppression reversed several conclusions.




```
# This is formatted as code
```

# Look-Ahead Variant

In [ ]:
"""Look-ahead variant: buffers L frames so the smoother can see the future."""
from collections import deque
import math, time
import numpy as np
import cv2


class LookaheadStabilizer(OnlineStabilizer):
    def __init__(self, cfg=StabConfig(), cam_id=0, lookahead=5, sigma=None):
        super().__init__(cfg, cam_id)
        self.L = int(lookahead)
        self.sigma = sigma if sigma else max(self.L / 2.0, 1.0)
        self.R = max(int(round(3 * self.sigma)), self.L, 1)
        self._buf = deque()
        self._hist = deque(maxlen=self.R + self.L + 1)

    def _weights(self, n, center):
        idx = np.arange(n) - center
        w = np.exp(-0.5 * (idx / self.sigma) ** 2)
        return w / w.sum()

    def process(self, frame, external_correction=None):
        t0 = time.perf_counter()
        cfg = self.cfg
        gray = self._prepare(frame.image)
        motion = Motion()
        need = (self.prev_pts is None or len(self.prev_pts) < cfg.min_tracks
                or self.frame_count % cfg.redetect_interval == 0)
        if self.prev_gray is not None:
            if need:
                self.prev_pts = self._detect(self.prev_gray, keep=self.prev_pts)
            if self.prev_pts is not None and len(self.prev_pts) >= 6:
                p0, p1 = self._track(self.prev_gray, gray, self.prev_pts)
                if p0 is not None:
                    motion = self._estimate(p0, p1)
                    self.prev_pts = p1.reshape(-1, 1, 2).astype(np.float32)
                else:
                    self.prev_pts = None
            else:
                self.prev_pts = None
        if motion.valid:
            w = min(1.0, motion.n_inliers / float(cfg.conf_inliers))
            self.traj += w * motion.as_array()
        self.prev_gray = gray
        self.frame_count += 1
        t_est = (time.perf_counter() - t0) * 1000.0
        self._hist.append(self.traj.copy())
        self._buf.append((frame, motion, t_est, self.traj.copy()))
        if len(self._buf) <= self.L:
            return None
        return self._emit()

    def _emit(self):
        t1 = time.perf_counter()
        frame, motion, t_est, traj_i = self._buf.popleft()
        center = max(min(len(self._hist) - 1 - len(self._buf), len(self._hist) - 1), 0)
        W, H = self.size
        mx = self.cfg.crop_ratio * W * 0.95
        my = self.cfg.crop_ratio * H * 0.95
        T = np.asarray(self._hist)
        smooth = (T * self._weights(len(T), center)[:, None]).sum(axis=0)
        corr = smooth - traj_i
        ma = math.radians(self.cfg.max_angle_deg)
        corr = self._soft_limit(corr, mx, my, ma)
        M = self._warp_matrix(corr)
        for _ in range(6):
            if self._fits(M, W, H): break
            corr *= 0.85
            M = self._warp_matrix(corr)
        out = cv2.warpAffine(frame.image, M, (W, H),
                             flags=self.cfg.interpolation,
                             borderMode=self.cfg.border_mode)
        return StabilizedFrame(out, frame.timestamp, frame.index, frame.cam_id,
                               corr, motion, M,
                               0 if self.prev_pts is None else len(self.prev_pts),
                               t_est + (time.perf_counter() - t1) * 1000.0)

    def flush(self):
        while self._buf:
            yield self._emit()

In [ ]:
def run_lookahead(path, cfg, L, sigma, n, out="la_tmp.mp4"):
    src = VideoFileSource(path)
    st = LookaheadStabilizer(cfg, lookahead=L, sigma=sigma)
    W, H = src.width, src.height
    w = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"mp4v"), src.fps, (W, H))
    times, corrs, i = [], [], 0
    while i < n:
        f = src.read()
        if f is None: break
        sf = st.process(f)                     # tampon dolana kadar None
        if sf is not None:
            times.append(sf.proc_ms); corrs.append(sf.correction.copy())
            w.write(sf.image)
        i += 1
    for sf in st.flush():                      # sonda tamponu boşalt
        times.append(sf.proc_ms); corrs.append(sf.correction.copy())
        w.write(sf.image)
    w.release(); src.release()
    return out, np.array(times), np.array(corrs)


def hf_of_output(path, out, cfg, corrs, n):
    z = 1/(1-2*cfg.crop_ratio)
    a = _highpass(_trajectory(path, n))
    b = _highpass(_trajectory(out, n)) / z
    c = cv2.VideoCapture(path); W, H = int(c.get(3)), int(c.get(4)); c.release()
    mx, my = cfg.crop_ratio*W*0.95, cfg.crop_ratio*H*0.95
    sat = max((np.abs(corrs[:,0]) >= mx*.98).mean(),
              (np.abs(corrs[:,1]) >= my*.98).mean())
    return (100*(1-np.std(b[:,0])/np.std(a[:,0])),
            100*(1-np.std(b[:,1])/np.std(a[:,1])), 100*sat)


P = CLIPS["meshflow"]
N = 150

print("causal")
best_c = None
for al in (0.85, 0.92, 0.95, 0.97):
    cfg = StabConfig(smooth_alpha=al, crop_ratio=0.08,
                     adaptive=False, max_angle_deg=3.0)
    x, y, s = hf_score(P, cfg, N)
    print(f"  alpha={al:<5}                x={x:5.1f}  y={y:5.1f}  sat={s:3.0f}%")
    if best_c is None or x+y > best_c[0]:
        best_c = (x+y, al, x, y)

print("\nlook-ahead")
cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                 adaptive=False, max_angle_deg=3.0)
best_l, LA = {}, {}
for L in (2, 3):
    for sg in (3, 5, 8, 12):
        out, t, co = run_lookahead(P, cfg, L, sg, N, f"la_{L}_{sg}.mp4")
        x, y, s = hf_of_output(P, out, cfg, co, N)
        if L not in best_l or x+y > best_l[L][0]:
            best_l[L] = (x+y, sg, x, y, s, t.mean()); LA[L] = out
for L, (tot, sg, x, y, s, ms) in sorted(best_l.items()):
    print(f"  L={L} ({L*1000//30:3d} ms) sigma={sg:<3}      "
          f"x={x:5.1f}  y={y:5.1f}  sat={s:3.0f}%  ms={ms:.2f}")

bl = max(best_l.items(), key=lambda kv: kv[1][0])
print(f"\nbest causal    : alpha={best_c[1]}  x={best_c[2]:.1f}  y={best_c[3]:.1f}")
print(f"best look-ahead: L={bl[0]} ({bl[0]*1000//30} ms)  "
      f"x={bl[1][2]:.1f}  y={bl[1][3]:.1f}")
print(f"gain           : x +{bl[1][2]-best_c[2]:.1f}  y +{bl[1][3]-best_c[3]:.1f}")

In [ ]:
ref = stabilize_clip(P, StabConfig(smooth_alpha=0.95, crop_ratio=0.08,
                                   adaptive=False, max_angle_deg=3.0),
                     out_dir="la_ref", max_frames=N, side_by_side=True)
print("causal, no delay")
display(show(ref["out"]))

In [ ]:
L = 3
SIGMA = 12

# ileri bakış çıktısını ham kareyle yan yana yaz
src = VideoFileSource(P)
st = LookaheadStabilizer(cfg, lookahead=L, sigma=SIGMA)
W, H = src.width, src.height
w = cv2.VideoWriter("la_sbs.mp4", cv2.VideoWriter_fourcc(*"mp4v"), src.fps, (W*2, H))

raws, i = [], 0
while i < N:
    f = src.read()
    if f is None: break
    raws.append(f.image)
    sf = st.process(f)
    if sf is not None:
        w.write(np.hstack([raws[len(raws)-1-L], sf.image]))   # L kare geriden eşle
    i += 1
for sf in st.flush():
    idx = len(raws) - len(list(st._buf)) - 1
    w.write(np.hstack([raws[max(idx, 0)], sf.image]))
w.release(); src.release()

!ffmpeg -y -loglevel error -i la_sbs.mp4 -vcodec libx264 -pix_fmt yuv420p la_sbs_h264.mp4
print(f"left: raw   right: look-ahead L={L} ({L*1000//30} ms delay)")
display(show("la_sbs_h264.mp4"))

In [ ]:
JETSON = {}
for name, path in {"jetson_1": "test_kayit4.mkv",
                   "jetson_2": "test_kayit3.mkv"}.items():
    if os.path.exists(path):
        JETSON[name] = path
    else:
        print(f"yok: {path}")
print(JETSON)

In [ ]:
CROP = 0.08
N = 300
BEST_OUT = {}          # sonuçlar buraya

for name, P2 in list(JETSON.items()):      # list() ile kopyala
    print(f"\n=== {name} ===")
    best_c = None
    for al in (0.85, 0.92, 0.95):
        c = StabConfig(smooth_alpha=al, crop_ratio=CROP,
                       adaptive=False, max_angle_deg=3.0)
        x, y, s = hf_score(P2, c, N)
        print(f"  causal alpha={al}      x={x:5.1f}  y={y:5.1f}  sat={s:3.0f}%")
        if best_c is None or x+y > best_c[0]: best_c = (x+y, al, x, y)

    c = StabConfig(smooth_alpha=0.92, crop_ratio=CROP,
                   adaptive=False, max_angle_deg=3.0)
    best_l = None
    for L in (2, 3, 5):
        out, t, co = run_lookahead(P2, c, L, 12, N, f"{name}_la_{L}.mp4")
        x, y, s = hf_of_output(P2, out, c, co, N)
        print(f"  lookahead L={L} ({L*1000//30:3d} ms)  x={x:5.1f}  y={y:5.1f}  sat={s:3.0f}%")
        if best_l is None or x+y > best_l[0]: best_l = (x+y, L, x, y, out)

    print(f"  -> gain: x {best_l[2]-best_c[2]:+.1f}  y {best_l[3]-best_c[3]:+.1f}")
    BEST_OUT[name] = best_l[4]

# şerit tabanlı test


In [ ]:
"""
Strip-based stabilizer: one correction per horizontal band instead of one for
the whole frame.

Why: a global affine transform applies the same shift everywhere, so it cannot
undo distortion that varies *within* the frame. The quadrant test showed exactly
that -- on handheld footage the four quadrants disagreed more than they agreed.
The usual cause is rolling shutter: the sensor reads the frame row by row, so
the top and bottom of one frame are captured a few milliseconds apart, and if
the camera moves in between the image shears.

Splitting into horizontal bands and estimating each one separately can undo that
shear. It does not help with parallax, which needs a full 2D mesh.

Cost: features get divided among the strips, so each estimate is noisier. Strips
with little texture may fail entirely, in which case they fall back to the
global motion.
"""
import math
import numpy as np
import cv2


class StripStabilizer(OnlineStabilizer):
    def __init__(self, cfg=StabConfig(), cam_id=0, n_strips=4,
                 strip_alpha=None, min_strip_pts=12, blend=True):
        """
        n_strips       : horizontal bands (4-8 is sensible; more = noisier)
        strip_alpha    : smoothing for the per-strip residual. Defaults to the
                         global alpha. The residual is small and fast, so a
                         lower value tracks it better.
        min_strip_pts  : below this many inliers a strip falls back to global
        blend          : interpolate between strip centres instead of hard bands
        """
        super().__init__(cfg, cam_id)
        self.n_strips = int(n_strips)
        self.strip_alpha = cfg.smooth_alpha if strip_alpha is None else strip_alpha
        self.min_strip_pts = min_strip_pts
        self.blend = blend
        self._strip_traj = np.zeros((self.n_strips, 2))
        self._strip_smooth = np.zeros((self.n_strips, 2))

    def _strip_motions(self, p0, p1, gh):
        """Per-strip translation, measured on top of the global motion."""
        out = np.zeros((self.n_strips, 2))
        ok = np.zeros(self.n_strips, bool)
        if p0 is None or len(p0) < self.min_strip_pts:
            return out, ok
        a = p0.reshape(-1, 2)
        b = p1.reshape(-1, 2)
        edges = np.linspace(0, gh, self.n_strips + 1)
        for i in range(self.n_strips):
            m = (a[:, 1] >= edges[i]) & (a[:, 1] < edges[i + 1])
            if m.sum() < self.min_strip_pts:
                continue
            d = b[m] - a[m]
            # median is robust enough here and much cheaper than another RANSAC
            out[i] = np.median(d, axis=0) / self.scale
            ok[i] = True
        return out, ok

    def _strip_correction(self, H):
        """Per-row correction, on top of the global one."""
        corr = self._strip_smooth - self._strip_traj
        # keep it small; this is a shear correction, not a second stabilizer
        lim = self.cfg.crop_ratio * H * 0.25
        corr = np.clip(corr, -lim, lim)
        centres = (np.arange(self.n_strips) + 0.5) * (H / self.n_strips)
        rows = np.arange(H)
        if self.blend:
            dx = np.interp(rows, centres, corr[:, 0])
            dy = np.interp(rows, centres, corr[:, 1])
        else:
            idx = np.minimum((rows * self.n_strips // H), self.n_strips - 1)
            dx, dy = corr[idx, 0], corr[idx, 1]
        return dx, dy

    def process(self, frame: Frame, external_correction=None) -> StabilizedFrame:
        import time
        t0 = time.perf_counter()
        cfg = self.cfg
        gray = self._prepare(frame.image)
        W, H = self.size
        mx, my = cfg.crop_ratio * W * 0.95, cfg.crop_ratio * H * 0.95

        motion = Motion()
        p0 = p1 = None
        need = (self.prev_pts is None or len(self.prev_pts) < cfg.min_tracks
                or self.frame_count % cfg.redetect_interval == 0)
        if self.prev_gray is not None:
            if need:
                self.prev_pts = self._detect(self.prev_gray, keep=self.prev_pts)
            if self.prev_pts is not None and len(self.prev_pts) >= 6:
                p0, p1 = self._track(self.prev_gray, gray, self.prev_pts)
                if p0 is not None:
                    motion = self._estimate(p0, p1)
                    self.prev_pts = p1.reshape(-1, 1, 2).astype(np.float32)
                else:
                    self.prev_pts = None
            else:
                self.prev_pts = None

        if motion.valid:
            w = min(1.0, motion.n_inliers / float(cfg.conf_inliers))
            self.traj += w * motion.as_array()

        # --- per-strip residual, relative to the global motion ---
        sm, ok = self._strip_motions(p0, p1, gray.shape[0])
        g = motion.as_array()[:2] if motion.valid else np.zeros(2)
        a = self.strip_alpha
        for i in range(self.n_strips):
            if ok[i]:
                self._strip_traj[i] += sm[i] - g          # residual only
            self._strip_smooth[i] = a * self._strip_smooth[i] + (1 - a) * self._strip_traj[i]

        corr = self._smooth_step(mx, my)
        if external_correction is not None:
            corr = np.asarray(external_correction, dtype=np.float64)

        M = self._warp_matrix(corr)
        for _ in range(6):
            if self._fits(M, W, H):
                break
            corr *= 0.85
            M = self._warp_matrix(corr)

        out = cv2.warpAffine(frame.image, M, (W, H),
                             flags=cfg.interpolation, borderMode=cfg.border_mode)

        # --- row-wise shear correction on top ---
        dx, dy = self._strip_correction(H)
        if np.abs(dx).max() > 0.3 or np.abs(dy).max() > 0.3:
            map_x = np.tile(np.arange(W, dtype=np.float32), (H, 1)) - dx[:, None].astype(np.float32)
            map_y = (np.tile(np.arange(H, dtype=np.float32), (W, 1)).T
                     - dy[:, None].astype(np.float32))
            out = cv2.remap(out, map_x, map_y, cfg.interpolation,
                            borderMode=cfg.border_mode)

        self.prev_gray = gray
        self.frame_count += 1
        return StabilizedFrame(out, frame.timestamp, frame.index, frame.cam_id,
                               corr, motion, M,
                               0 if self.prev_pts is None else len(self.prev_pts),
                               (time.perf_counter() - t0) * 1000.0)

In [ ]:
def run_strips(path, cfg, n_strips, n=300, out="strip_out.mp4"):
    src = VideoFileSource(path)
    st = StripStabilizer(cfg, n_strips=n_strips)
    W, H = src.width, src.height
    w = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"mp4v"), src.fps, (W, H))
    t, i = [], 0
    while i < n:
        f = src.read()
        if f is None: break
        sf = st.process(f); t.append(sf.proc_ms); w.write(sf.image); i += 1
    w.release(); src.release()
    return out, np.array(t)


P = CLIPS["running"]          # ya da JETSON_CLIPS["jetson_1"]
N = 300
cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                 adaptive=False, max_angle_deg=3.0)

x, y, s = hf_score(P, cfg, N)
print(f"global      HF x={x:5.1f}  y={y:5.1f}")

for ns in (4, 6, 8):
    out, t = run_strips(P, cfg, ns, N, f"strip_{ns}.mp4")
    z = 1/(1-2*cfg.crop_ratio)
    a = _highpass(_trajectory(P, N)); b = _highpass(_trajectory(out, N))/z
    print(f"şerit n={ns}   HF x={100*(1-np.std(b[:,0])/np.std(a[:,0])):5.1f}  "
          f"y={100*(1-np.std(b[:,1])/np.std(a[:,1])):5.1f}  ms={t.mean():.2f}")

In [ ]:
def run_strips_sbs(path, cfg, n_strips, n=300, out="strip_sbs.mp4"):
    src = VideoFileSource(path)
    st = StripStabilizer(cfg, n_strips=n_strips)
    W, H = src.width, src.height
    w = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"mp4v"), src.fps, (W*2, H))
    t, i = [], 0
    while i < n:
        f = src.read()
        if f is None: break
        sf = st.process(f)
        t.append(sf.proc_ms)
        w.write(np.hstack([f.image, sf.image]))     # sol ham, sağ şerit
        i += 1
    w.release(); src.release()
    return out, np.array(t)


out, t = run_strips_sbs(P, cfg, 4, N, "strip_sbs.mp4")
!ffmpeg -y -loglevel error -i strip_sbs.mp4 -vcodec libx264 -pix_fmt yuv420p strip_sbs_h264.mp4
print(f"sol: ham   sağ: şerit (n=4)   {t.mean():.2f} ms/kare")
display(show("strip_sbs_h264.mp4"))

# MESH

In [ ]:
"""
Mesh-based stabilizer: a grid of local motions instead of one global transform.

Why this exists: a single affine transform applies the same correction to the
whole frame, so distortion that varies *within* the frame is out of reach. The
quadrant test showed exactly that -- on handheld footage the four quadrants
disagreed with each other more than they agreed. Rolling shutter and parallax
both do this. A strip-based version (rows only) was tried first and gained
nothing, which suggested the variation is genuinely two-dimensional.

How it works, following the shape of MeshFlow (Liu et al., ECCV 2016):

  1. estimate the global motion as usual -- this is the backbone
  2. assign each tracked feature to a grid cell and take the median residual
     (feature motion minus global motion) per cell
  3. cells with too few features inherit from their neighbours, then the whole
     grid is smoothed spatially -- without this the mesh tears
  4. smooth each vertex over time with its own EMA
  5. warp with a per-cell displacement map on top of the global warp

Cost is a per-pixel remap plus the grid bookkeeping. Expect roughly 1.5-2x the
global version.
"""
import math
import time

import numpy as np
import cv2



class MeshStabilizer(OnlineStabilizer):
    def __init__(self, cfg=StabConfig(), cam_id=0,
                 grid=(4, 4), mesh_alpha=None, min_cell_pts=6,
                 spatial_sigma=1.0, max_local_ratio=0.30):
        """
        grid            : (rows, cols) of mesh cells. 4x4 is a reasonable start;
                          finer grids get noisy because features per cell drop.
        mesh_alpha      : temporal smoothing of the local residual. Defaults to
                          the global alpha.
        min_cell_pts    : below this a cell has no measurement of its own and
                          gets filled in from its neighbours.
        spatial_sigma   : blur applied across the grid. This is what stops the
                          mesh from tearing at cell boundaries.
        max_local_ratio : local correction ceiling, as a fraction of the crop
                          budget. Local motion is a *residual* -- if it grows
                          large something has gone wrong, so it stays small.
        """
        super().__init__(cfg, cam_id)
        self.gr, self.gc = grid
        self.mesh_alpha = cfg.smooth_alpha if mesh_alpha is None else mesh_alpha
        self.min_cell_pts = min_cell_pts
        self.spatial_sigma = spatial_sigma
        self.max_local_ratio = max_local_ratio
        self._cell_traj = np.zeros((self.gr, self.gc, 2))
        self._cell_smooth = np.zeros((self.gr, self.gc, 2))
        self._map_cache = None

    # ---------------- per-cell measurement ----------------

    def _cell_motions(self, p0, p1, gh, gw, global_xy):
        """Median residual motion per cell, in full-resolution pixels."""
        res = np.zeros((self.gr, self.gc, 2))
        seen = np.zeros((self.gr, self.gc), bool)
        if p0 is None or len(p0) < self.min_cell_pts:
            return res, seen

        a = p0.reshape(-1, 2)
        d = (p1.reshape(-1, 2) - a) / self.scale - global_xy   # residual
        ri = np.clip((a[:, 1] * self.gr / gh).astype(int), 0, self.gr - 1)
        ci = np.clip((a[:, 0] * self.gc / gw).astype(int), 0, self.gc - 1)

        for r in range(self.gr):
            for c in range(self.gc):
                m = (ri == r) & (ci == c)
                if m.sum() >= self.min_cell_pts:
                    res[r, c] = np.median(d[m], axis=0)
                    seen[r, c] = True
        return res, seen

    def _fill_and_smooth(self, res, seen):
        """Fill empty cells from neighbours, then blur across the grid.

        Cells with no features would otherwise sit at zero while their
        neighbours move, and the warp would tear along that seam.
        """
        if not seen.any():
            return np.zeros_like(res)
        out = res.copy()
        # iterative neighbour fill
        for _ in range(max(self.gr, self.gc)):
            missing = ~seen
            if not missing.any():
                break
            filled = seen.copy()
            for r, c in zip(*np.where(missing)):
                vals = []
                for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                    rr, cc = r + dr, c + dc
                    if 0 <= rr < self.gr and 0 <= cc < self.gc and seen[rr, cc]:
                        vals.append(res[rr, cc])
                if vals:
                    out[r, c] = np.mean(vals, axis=0)
                    filled[r, c] = True
            res, seen = out.copy(), filled
        if self.spatial_sigma > 0:
            k = max(3, int(2 * round(self.spatial_sigma) + 1))
            for ch in range(2):
                out[:, :, ch] = cv2.GaussianBlur(
                    out[:, :, ch].astype(np.float32), (k, k), self.spatial_sigma)
        return out

    # ---------------- warp map ----------------

    def _mesh_map(self, W, H):
        """Per-pixel displacement, interpolated from the cell grid."""
        corr = self._cell_smooth - self._cell_traj
        lim = self.cfg.crop_ratio * min(W, H) * self.max_local_ratio
        corr = np.clip(corr, -lim, lim)
        if np.abs(corr).max() < 0.3:
            return None

        # bilinear upsample of the coarse grid to full resolution
        dx = cv2.resize(corr[:, :, 0].astype(np.float32), (W, H),
                        interpolation=cv2.INTER_LINEAR)
        dy = cv2.resize(corr[:, :, 1].astype(np.float32), (W, H),
                        interpolation=cv2.INTER_LINEAR)
        if self._map_cache is None or self._map_cache[0].shape != (H, W):
            gx, gy = np.meshgrid(np.arange(W, dtype=np.float32),
                                 np.arange(H, dtype=np.float32))
            self._map_cache = (gx, gy)
        gx, gy = self._map_cache
        return (gx - dx), (gy - dy)

    # ---------------- main ----------------

    def process(self, frame: Frame, external_correction=None) -> StabilizedFrame:
        t0 = time.perf_counter()
        cfg = self.cfg
        gray = self._prepare(frame.image)
        gh, gw = gray.shape
        W, H = self.size
        mx, my = cfg.crop_ratio * W * 0.95, cfg.crop_ratio * H * 0.95

        motion = Motion()
        p0 = p1 = None
        need = (self.prev_pts is None or len(self.prev_pts) < cfg.min_tracks
                or self.frame_count % cfg.redetect_interval == 0)
        if self.prev_gray is not None:
            if need:
                self.prev_pts = self._detect(self.prev_gray, keep=self.prev_pts)
            if self.prev_pts is not None and len(self.prev_pts) >= 6:
                p0, p1 = self._track(self.prev_gray, gray, self.prev_pts)
                if p0 is not None:
                    motion = self._estimate(p0, p1)
                    self.prev_pts = p1.reshape(-1, 1, 2).astype(np.float32)
                else:
                    self.prev_pts = None
            else:
                self.prev_pts = None

        if motion.valid:
            w = min(1.0, motion.n_inliers / float(cfg.conf_inliers))
            self.traj += w * motion.as_array()

        # local residual grid
        gxy = motion.as_array()[:2] if motion.valid else np.zeros(2)
        res, seen = self._cell_motions(p0, p1, gh, gw, gxy)
        res = self._fill_and_smooth(res, seen)
        self._cell_traj += res
        a = self.mesh_alpha
        self._cell_smooth = a * self._cell_smooth + (1 - a) * self._cell_traj

        # global correction, unchanged
        corr = self._smooth_step(mx, my)
        if external_correction is not None:
            corr = np.asarray(external_correction, dtype=np.float64)

        M = self._warp_matrix(corr)
        for _ in range(6):
            if self._fits(M, W, H):
                break
            corr *= 0.85
            M = self._warp_matrix(corr)

        out = cv2.warpAffine(frame.image, M, (W, H),
                             flags=cfg.interpolation, borderMode=cfg.border_mode)

        mp = self._mesh_map(W, H)
        if mp is not None:
            out = cv2.remap(out, mp[0], mp[1], cfg.interpolation,
                            borderMode=cfg.border_mode)

        self.prev_gray = gray
        self.frame_count += 1
        return StabilizedFrame(out, frame.timestamp, frame.index, frame.cam_id,
                               corr, motion, M,
                               0 if self.prev_pts is None else len(self.prev_pts),
                               (time.perf_counter() - t0) * 1000.0)

In [ ]:
def run_mesh(path, cfg, grid, n=300, out="mesh_out.mp4", sbs=False):
    src = VideoFileSource(path)
    st = MeshStabilizer(cfg, grid=grid)
    W, H = src.width, src.height
    w = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"mp4v"), src.fps,
                        (W*2 if sbs else W, H))
    t, i = [], 0
    while i < n:
        f = src.read()
        if f is None: break
        sf = st.process(f); t.append(sf.proc_ms)
        w.write(np.hstack([f.image, sf.image]) if sbs else sf.image)
        i += 1
    w.release(); src.release()
    return out, np.array(t)


P = CLIPS["running"]
N = 300
cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                 adaptive=False, max_angle_deg=3.0)
z = 1/(1-2*cfg.crop_ratio)

x, y, s = hf_score(P, cfg, N)
print(f"global      HF x={x:5.1f}  y={y:5.1f}")

for g in [(3,3), (4,4), (6,6)]:
    out, t = run_mesh(P, cfg, g, N, f"mesh_{g[0]}.mp4")
    a = _highpass(_trajectory(P, N)); b = _highpass(_trajectory(out, N))/z
    print(f"mesh {g[0]}x{g[1]}    HF x={100*(1-np.std(b[:,0])/np.std(a[:,0])):5.1f}  "
          f"y={100*(1-np.std(b[:,1])/np.std(a[:,1])):5.1f}  ms={t.mean():.2f}")

In [ ]:
def run_compare(path, cfg, grid=(3,3), n=300, out="cmp3.mp4"):
    """sol: ham | orta: global | sağ: mesh"""
    src = VideoFileSource(path)
    g_st = OnlineStabilizer(cfg)
    m_st = MeshStabilizer(cfg, grid=grid)
    W, H = src.width, src.height
    w = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"mp4v"), src.fps, (W*3, H))
    tg, tm, i = [], [], 0
    while i < n:
        f = src.read()
        if f is None: break
        raw = f.image.copy()
        gs = g_st.process(Frame(raw.copy(), f.timestamp, f.index))
        ms = m_st.process(Frame(raw.copy(), f.timestamp, f.index))
        tg.append(gs.proc_ms); tm.append(ms.proc_ms)
        w.write(np.hstack([raw, gs.image, ms.image]))
        i += 1
    w.release(); src.release()
    print(f"global {np.mean(tg):.2f} ms   mesh {np.mean(tm):.2f} ms")
    return out


out = run_compare(CLIPS["running"], cfg, grid=(3,3), n=300)
!ffmpeg -y -loglevel error -i cmp3.mp4 -vcodec libx264 -pix_fmt yuv420p cmp3_h264.mp4
print("sol: ham | orta: global | sağ: mesh")
display(show("cmp3_h264.mp4", width=1300))

In [ ]:
def run_pair(path, cfg, grid=(3,3), n=300, out="pair.mp4"):
    """sol: global | sağ: mesh"""
    src = VideoFileSource(path)
    g_st = OnlineStabilizer(cfg)
    m_st = MeshStabilizer(cfg, grid=grid)
    W, H = src.width, src.height
    w = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"mp4v"), src.fps, (W*2, H))
    i = 0
    while i < n:
        f = src.read()
        if f is None: break
        raw = f.image.copy()
        gs = g_st.process(Frame(raw.copy(), f.timestamp, f.index))
        ms = m_st.process(Frame(raw.copy(), f.timestamp, f.index))
        w.write(np.hstack([gs.image, ms.image]))
        i += 1
    w.release(); src.release()
    return out


out = run_pair(CLIPS["running"], cfg, grid=(3,3), n=300)
!ffmpeg -y -loglevel error -i pair.mp4 -vcodec libx264 -pix_fmt yuv420p pair_h264.mp4
print("sol: global | sağ: mesh")
display(show("pair_h264.mp4"))

In [ ]:
cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                 adaptive=False, max_angle_deg=3.0)
z = 1/(1-2*cfg.crop_ratio)
N = 300

ALL = dict(CLIPS)
try:
    ALL.update(JETSON_CLIPS)
except NameError:
    pass

MESH_OUT = {}
for name, P in ALL.items():
    print(f"\n=== {name} ===")
    x, y, s = hf_score(P, cfg, N)
    print(f"  global      HF x={x:5.1f}  y={y:5.1f}")
    for g in [(3,3), (4,4)]:
        out, t = run_mesh(P, cfg, g, N, f"mesh_{name}_{g[0]}.mp4")
        a = _highpass(_trajectory(P, N)); b = _highpass(_trajectory(out, N))/z
        mx = 100*(1-np.std(b[:,0])/np.std(a[:,0]))
        my = 100*(1-np.std(b[:,1])/np.std(a[:,1]))
        print(f"  mesh {g[0]}x{g[1]}    HF x={mx:5.1f}  y={my:5.1f}  ms={t.mean():.2f}")
        if g == (3,3):
            MESH_OUT[name] = out

In [ ]:
KLIP = "running"        # ismi değiştirerek diğerlerine bak

out = run_pair(ALL[KLIP], cfg, grid=(3,3), n=N, out=f"pair_{KLIP}.mp4")
!ffmpeg -y -loglevel error -i {out} -vcodec libx264 -pix_fmt yuv420p pair_h264.mp4
print(f"{KLIP} — sol: global | sağ: mesh")
display(show("pair_h264.mp4"))

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def motion_split(path, n=300):
    cum = _trajectory(path, n)
    def lowpass(x, w=31):
        pad = np.pad(x, ((w//2, w//2), (0, 0)), mode="edge")
        return sliding_window_view(pad, w, axis=0).mean(axis=-1)
    lp = lowpass(cum); hp = cum - lp
    print(f"{os.path.basename(path)}")
    for i, lbl in enumerate(["x", "y"]):
        low, high = np.std(lp[:, i]), np.std(hp[:, i])
        print(f"  {lbl}: kasıtlı={low:7.1f}px  sarsıntı={high:6.1f}px"
              f"  -> sarsıntı enerjinin %{100*high**2/(high**2+low**2):.0f}'ü")

for n in ("meshflow", "running"):
    motion_split(CLIPS[n], 300)

In [ ]:
# =====================================================================
# Teşhis hücresi — A1 ve A2 çalıştırıldıktan sonra yapıştır.
#
# Parametre taramadan ÖNCE bu çalışmalı. Sorunun hareket kestiriminde mi
# (izleme kopuyor / güven ağırlığı hareketi kırpıyor) yoksa limitleyicide mi
# (düzeltme kırpılıyor) olduğunu ayırır. İkisi tamamen farklı çözüm ister.
# =====================================================================
import numpy as np, math, cv2


def diagnose(path, cfg, n=300, label=""):
    st = OnlineStabilizer(cfg)
    src = VideoFileSource(path)
    W, H = src.width, src.height
    mx, my = cfg.crop_ratio * W * cfg.safety, cfg.crop_ratio * H * cfg.safety

    # --- _fits çağrı sayısını say: >1 ise küçültme döngüsü ateşlendi
    fits_n = {"k": 0}
    _fits0 = st._fits
    def fits(M, Wp, Hp):
        fits_n["k"] += 1
        return _fits0(M, Wp, Hp)
    st._fits = fits

    # --- FB kontrolünden kaç iz sağ çıkıyor
    _track0 = st._track
    trk = []
    def track(pg, cg, p0):
        r = _track0(pg, cg, p0)
        trk.append((len(p0), 0 if r[0] is None else len(r[0])))
        return r
    st._track = track

    valid, inl, wgts, shrink, ratios, motions = [], [], [], [], [], []
    i = 0
    while i < n:
        f = src.read()
        if f is None:
            break
        fits_n["k"] = 0
        sf = st.process(f)
        valid.append(sf.motion.valid)
        if sf.motion.valid:
            inl.append(sf.motion.n_inliers)
            wgts.append(min(1.0, sf.motion.n_inliers / float(cfg.conf_inliers)))
            motions.append(sf.motion.as_array())
        shrink.append(max(0, fits_n["k"] - 1))          # 0 = döngü hiç dönmedi
        ratios.append(max(abs(sf.correction[0]) / mx,
                          abs(sf.correction[1]) / my))
        i += 1
    src.release()

    valid = np.array(valid); shrink = np.array(shrink); ratios = np.array(ratios)
    inl = np.array(inl) if inl else np.array([0])
    wgts = np.array(wgts) if len(wgts) else np.array([0.0])
    tin = np.array(trk) if trk else np.zeros((1, 2))
    surv = tin[:, 1] / np.maximum(tin[:, 0], 1)

    print(f"\n===== {label or path}  ({W}x{H}, {i} kare) =====")
    print(f"  gecerli hareket kestirimi : {100*valid.mean():5.1f} %"
          f"   <- %90 altiysa sorun izlemede")
    print(f"  FB sonrasi hayatta kalan iz: {100*surv.mean():5.1f} %"
          f"   (medyan {100*np.median(surv):.0f} %)")
    print(f"  inlier                    : ort {inl.mean():5.1f}  "
          f"medyan {np.median(inl):5.1f}  p10 {np.percentile(inl,10):5.1f}")
    print(f"  guven agirligi wgt        : ort {wgts.mean():5.3f}  "
          f"wgt<1 olan kare: {100*(wgts<0.999).mean():4.1f} %"
          f"   <- BURASI ONEMLI")
    print(f"  kayip yol (bias)          : {100*(1-wgts.mean()):5.1f} %"
          f" hareket traj'a hic islenmiyor")
    print(f"  duzeltme / limit orani    : ort {ratios.mean():5.2f}  "
          f">0.6 (tanh dizi): {100*(ratios>0.6).mean():4.1f} %  "
          f">0.98: {100*(ratios>0.98).mean():4.1f} %")
    print(f"  _fits kucultme dongusu    : {100*(shrink>0).mean():5.1f} % karede "
          f"ateslendi, ort {shrink[shrink>0].mean() if (shrink>0).any() else 0:.1f} adim"
          f"   <- >%5 ise yumusak limitleyici devre disi")

    # --- titremenin frekans bandi: EMA kesim frekansi dogru yerde mi
    if len(motions) > 32:
        m = np.cumsum(np.array(motions), axis=0)
        fs = src.fps if hasattr(src, "fps") else 30.0
        for ax, name in ((0, "x"), (1, "y")):
            sig = m[:, ax] - m[:, ax].mean()
            P = np.abs(np.fft.rfft(sig * np.hanning(len(sig))))**2
            fr = np.fft.rfftfreq(len(sig), 1.0/30.0)
            k = np.argmax(P[1:]) + 1
            band = fr[1:][P[1:] > 0.2*P[1:].max()]
            print(f"  {name} titreme tepe frekansi : {fr[k]:5.2f} Hz   "
                  f"(-6dB bandi {band.min():.2f}-{band.max():.2f} Hz)")
    a = cfg.smooth_alpha
    print(f"  EMA kesim frekansi        : {30*(1-a)/(2*math.pi*a):5.2f} Hz "
          f"(alpha={a})")
    return dict(valid=valid, inliers=inl, wgt=wgts, shrink=shrink, ratio=ratios)


# --- her klip icin calistir, running'i digerleriyle karsilastir
DIAG = {}
for name, p in CLIPS.items():
    DIAG[name] = diagnose(p, CFG, n=MAX_FRAMES, label=name)

print("""
Nasil okunur
------------
* 'gecerli hareket kestirimi' running'de digerlerinden belirgin dusukse
  sorun filtre degil, izleme. fb_threshold ve proc_width'e bak.
* 'wgt<1 olan kare' running'de yuksekse traj gercek kamera yolunun altinda
  kaliyor demektir. Bu sistematik bir sapma; alpha ile duzelmez.
* '_fits kucultme dongusu' sik ateslenmisse duzeltme sert carpanla
  kirpiliyor ve _soft_limit'in tanh'i devreden cikiyor.
* Titreme tepe frekansi EMA kesim frekansinin cok uzerindeyse alpha
  taramasi frekans tepkisini degil, doyma davranisini olcuyor.
""")

In [ ]:
def inlier_probe(path, cfg, n=300):
    src = VideoFileSource(path); st = OnlineStabilizer(cfg)
    inl, low, invalid, i = [], 0, 0, 0
    while i < n:
        f = src.read()
        if f is None: break
        sf = st.process(f)
        if sf.motion.valid:
            inl.append(sf.motion.n_inliers)
            if sf.motion.n_inliers < cfg.conf_inliers: low += 1
        else:
            invalid += 1
        i += 1
    src.release()
    inl = np.array(inl)
    print(f"{os.path.basename(path):20s} inlier ort={inl.mean():6.1f} p10={np.percentile(inl,10):5.0f}  "
          f"wgt<1=%{100*low/i:.1f}  geçersiz=%{100*invalid/i:.1f}")

cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08, adaptive=False, max_angle_deg=3.0)
for n, p in CLIPS.items():
    inlier_probe(p, cfg)

In [ ]:
cfg_base = dict(smooth_alpha=0.92, crop_ratio=0.08,
                adaptive=False, max_angle_deg=3.0)
P = CLIPS["running"]
N = 300

for label, extra in [
    ("fb açık, eşik 1.0 (mevcut)", dict(use_fb_check=True,  fb_threshold=1.0)),
    ("fb açık, eşik 2.0",          dict(use_fb_check=True,  fb_threshold=2.0)),
    ("fb açık, eşik 4.0",          dict(use_fb_check=True,  fb_threshold=4.0)),
    ("fb KAPALI",                  dict(use_fb_check=False)),
]:
    cfg = StabConfig(**cfg_base, **extra)
    inlier_probe(P, cfg)
    x, y, s = hf_score(P, cfg, N)
    print(f"  -> {label:28s} HF x={x:5.1f}  y={y:5.1f}  sat=%{s:.0f}\n")

In [ ]:
cfg = StabConfig(**cfg_base, max_corners=400, use_fb_check=False)
inlier_probe(P, cfg)
x, y, s = hf_score(P, cfg, N)
print(f"  -> max_corners=400, fb kapalı  HF x={x:5.1f}  y={y:5.1f}")

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def motion_split(path, n=300):
    cum = _trajectory(path, n)
    def lowpass(x, w=31):
        pad = np.pad(x, ((w//2, w//2), (0, 0)), mode="edge")
        return sliding_window_view(pad, w, axis=0).mean(axis=-1)
    lp = lowpass(cum); hp = cum - lp
    print(f"{os.path.basename(path):16s}", end="")
    for i, lbl in enumerate(["x", "y"]):
        low, high = np.std(lp[:, i]), np.std(hp[:, i])
        print(f"  {lbl}: kasıtlı={low:6.1f}px sarsıntı={high:5.1f}px "
              f"(%{100*high**2/(high**2+low**2):.0f})", end="")
    print()

for n in ("meshflow", "ostrich", "running"):
    motion_split(CLIPS[n], 300)

In [ ]:
fps = 30.0
for name in ("meshflow", "running"):
    cum = _trajectory(CLIPS[name], 300)
    d = np.diff(cum, axis=0)
    print(f"\n{name}")
    for i, lbl in enumerate(["x", "y"]):
        sig = d[:, i] - d[:, i].mean()
        f = np.abs(np.fft.rfft(sig))
        fr = np.fft.rfftfreq(len(sig), 1/fps)
        band = lambda lo, hi: f[(fr >= lo) & (fr < hi)].sum()
        tot = f[1:].sum()
        print(f"  {lbl}: 0-1Hz %{100*band(0,1)/tot:.0f}  "
              f"1-3Hz %{100*band(1,3)/tot:.0f}  "
              f"3-8Hz %{100*band(3,8)/tot:.0f}  "
              f"8-15Hz %{100*band(8,15)/tot:.0f}")

In [ ]:
P = CLIPS["running"]
cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                 adaptive=False, max_angle_deg=3.0)
N = 300

# 1) _fits kaç karede ateşleniyor?
src = VideoFileSource(P); st = OnlineStabilizer(cfg)
fires = [0]
orig = st._fits
def counted(M, W, H):
    r = orig(M, W, H)
    if not r: fires[0] += 1
    return r
st._fits = counted
i = 0
while i < N:
    f = src.read()
    if f is None: break
    st.process(f); i += 1
src.release()
print(f"_fits ateşleme: {fires[0]} kez / {i} kare  (%{100*fires[0]/i:.1f})")

# 2) enjekte edilmiş KESKİN sarsıntı ne kadar bastırılıyor?
shaken, gtp = add_shake(P, amp=15.0, rot=2.0, max_frames=N)
base = stabilize_clip(P, cfg, out_dir="rb", max_frames=N,
                      side_by_side=False, to_h264=False, quiet=True)
res = stabilize_clip(shaken, cfg, out_dir="rs", max_frames=N,
                     side_by_side=False, to_h264=False, quiet=True)
evaluate_gt(gtp, res, baseline=base)

In [ ]:
P = CLIPS["running"]
cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                 adaptive=False, max_angle_deg=3.0)
x, y, s = hf_score(P, cfg, 300)
print(f"core_v3   HF x={x:5.1f}  y={y:5.1f}")

In [ ]:
for ma in (0.5, 1.0, 2.0, 3.0):
    cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                     adaptive=False, max_angle_deg=ma)
    x, y, s = hf_score(P, cfg, 300)
    print(f"max_angle={ma}   HF x={x:5.1f}  y={y:5.1f}")

In [ ]:
for ma in (0.3, 0.5, 0.8, 1.2, 1.5):
    cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                     adaptive=False, max_angle_deg=ma)
    x, y, s = hf_score(P, cfg, 300)
    print(f"max_angle={ma}   HF x={x:5.1f}  y={y:5.1f}  sat=%{s:.0f}")

In [ ]:
for n in ("meshflow", "ostrich", "running"):
    for ma in (0.5, 1.0, 3.0):
        cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                         adaptive=False, max_angle_deg=ma)
        x, y, s = hf_score(CLIPS[n], cfg, 300)
        print(f"{n:10s} max_angle={ma}  HF x={x:5.1f}  y={y:5.1f}")

In [ ]:
CFG = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                 adaptive=False, max_angle_deg=0.5)
RESULTS = run_all(CLIPS, CFG, 300)
summary_table(RESULTS)

In [ ]:
print("core_v3 yüklü mü:", hasattr(OnlineStabilizer, "_fit_scale"))
print("guven duzeltmesi:", "_last_m" in OnlineStabilizer().__dict__ if False else
      hasattr(OnlineStabilizer(StabConfig()), "_last_m"))

In [ ]:
import time
P = CLIPS["meshflow"]
print("fit_scale var mı:", hasattr(OnlineStabilizer, "_fit_scale"))

for ma in (0.5, 1.0, 1.5, 3.0):
    cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                     adaptive=False, max_angle_deg=ma)
    r = stabilize_clip(P, cfg, out_dir="t", max_frames=200,
                       side_by_side=False, to_h264=False, quiet=True)
    x, y, s = hf_score(P, cfg, 200)
    print(f"max_angle={ma}  HF x={x:5.1f} y={y:5.1f}  ms={r['ms']:6.2f}  p95={r['ms_p95']:6.2f}")

In [ ]:
for ma in (0.5, 1.0, 1.5):
    cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                     adaptive=False, max_angle_deg=ma)
    R = run_all({"running": CLIPS["running"]}, cfg, 300)
    x, y, s = hf_score(CLIPS["running"], cfg, 300)
    r = R["running"]
    print(f"max_angle={ma}  HF x={x:5.1f} y={y:5.1f} | uçtan uca ötl={r['dxy_red']:5.1f}% "
          f"dön={r['da_red']:5.1f}% | ms={r['ms']:.1f}")

In [ ]:
CFG = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                 adaptive=False, max_angle_deg=1.0)
RESULTS = run_all(CLIPS, CFG, 300)
summary_table(RESULTS)

In [ ]:
old = stabilize_clip(CLIPS["running"],
                     StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                                adaptive=False, max_angle_deg=3.0),
                     out_dir="cmp_old", max_frames=300, side_by_side=False,
                     to_h264=False, quiet=True)
new = stabilize_clip(CLIPS["running"],
                     StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                                adaptive=False, max_angle_deg=1.0),
                     out_dir="cmp_new", max_frames=300, side_by_side=False,
                     to_h264=False, quiet=True)

co = cv2.VideoCapture(old["out"]); cn = cv2.VideoCapture(new["out"])
W = int(co.get(3)); H = int(co.get(4))
w = cv2.VideoWriter("angle_cmp.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 30, (W*2, H))
while True:
    a, fa = co.read(); b, fb = cn.read()
    if not (a and b): break
    w.write(np.hstack([fa, fb]))
w.release(); co.release(); cn.release()

!ffmpeg -y -loglevel error -i angle_cmp.mp4 -vcodec libx264 -pix_fmt yuv420p angle_cmp_h264.mp4
print("sol: max_angle=3.0 (eski)   sağ: max_angle=1.0 (yeni)")
display(show("angle_cmp_h264.mp4"))

In [ ]:
old = stabilize_clip(CLIPS["ostrich"],
                     StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                                adaptive=False, max_angle_deg=3.0),
                     out_dir="cmp_old", max_frames=300, side_by_side=False,
                     to_h264=False, quiet=True)
new = stabilize_clip(CLIPS["ostrich"],
                     StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                                adaptive=False, max_angle_deg=1.0),
                     out_dir="cmp_new", max_frames=300, side_by_side=False,
                     to_h264=False, quiet=True)

co = cv2.VideoCapture(old["out"]); cn = cv2.VideoCapture(new["out"])
W = int(co.get(3)); H = int(co.get(4))
w = cv2.VideoWriter("angle_cmp.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 30, (W*2, H))
while True:
    a, fa = co.read(); b, fb = cn.read()
    if not (a and b): break
    w.write(np.hstack([fa, fb]))
w.release(); co.release(); cn.release()

!ffmpeg -y -loglevel error -i angle_cmp.mp4 -vcodec libx264 -pix_fmt yuv420p angle_cmp_h264.mp4
print("sol: max_angle=3.0 (eski)   sağ: max_angle=1.0 (yeni)")
display(show("angle_cmp_h264.mp4"))

In [ ]:
P = CLIPS["running"]
for ma in (1.0, 3.0):
    cfg = StabConfig(smooth_alpha=0.92, crop_ratio=0.08,
                     adaptive=False, max_angle_deg=ma)
    R = run_all({"running": P}, cfg, 300)
    x, y, s = hf_score(P, cfg, 300)
    r = R["running"]
    print(f"max_angle={ma}  HF x={x:5.1f} y={y:5.1f} | ötl={r['dxy_red']:5.1f}% "
          f"dön={r['da_red']:5.1f}% | ms={r['ms']:.1f}")

In [ ]:
print(hasattr(OnlineStabilizer, "_fit_scale"))